In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Clone Repo & Install Dependencies
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, os

if not os.path.exists("/kaggle/working/disaster_decision_support"):
    subprocess.run([
        "git", "clone",
        "https://github.com/karthiksekar1821/disaster_decision_support.git",
        "/kaggle/working/disaster_decision_support"
    ], check=True)

# Install only what is needed (captum installed later in Cell 7)
subprocess.run(["pip", "install", "-q", "gradio", "joblib"], check=True)

print("✅ Repository cloned and dependencies installed.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Configure All Paths
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, numpy as np, json

BASE_INPUT    = "/kaggle/input/notebooks/karthiksekar1821/main-project"
OUTPUT_DIR    = f"{BASE_INPUT}/output"
DATA_DIR      = "/kaggle/input/datasets/karthiksekar1821/humaid"
PROCESSED_DIR = f"{BASE_INPUT}/disaster_decision_support/data/processed"
RESULTS_DIR   = "/kaggle/working/results"

os.makedirs(RESULTS_DIR, exist_ok=True)

sys.path.insert(0, "/kaggle/working/disaster_decision_support/src/training")
sys.path.insert(0, "/kaggle/working/disaster_decision_support/src/analysis")
sys.path.insert(0, "/kaggle/working/disaster_decision_support/src/evaluation")
sys.path.insert(0, "/kaggle/working/disaster_decision_support/src/data")
sys.path.insert(0, "/kaggle/working/disaster_decision_support/src/app")

print("Paths configured:")
print(f"  BASE_INPUT:    {BASE_INPUT}")
print(f"  OUTPUT_DIR:    {OUTPUT_DIR}")
print(f"  DATA_DIR:      {DATA_DIR}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  RESULTS_DIR:   {RESULTS_DIR}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Load All Predictions + Train Texts
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np, json, pandas as pd

def load_predictions(model_key, split="test"):
    """Load predictions trying seed_42 path first, then non-seed path."""
    path_with_seed    = f"{OUTPUT_DIR}/{model_key}/seed_42/predictions/{split}_predictions.npz"
    path_without_seed = f"{OUTPUT_DIR}/{model_key}/predictions/{split}_predictions.npz"

    if os.path.exists(path_with_seed):
        data = np.load(path_with_seed)
        print(f"  ✅ {model_key}/{split} loaded from seed_42 path")
    elif os.path.exists(path_without_seed):
        data = np.load(path_without_seed)
        print(f"  ✅ {model_key}/{split} loaded from non-seed path")
    else:
        raise FileNotFoundError(
            f"No predictions found for {model_key} ({split})\n"
            f"  Checked: {path_with_seed}\n"
            f"  Checked: {path_without_seed}"
        )
    return {"preds": data["preds"], "labels": data["labels"], "probs": data["probs"]}

# Load predictions from all 8 models
model_keys = ["roberta", "deberta", "electra", "bert", "bertweet", "xtremedistil", "cnn", "bilstm"]
print("Loading validation predictions...")
val_preds  = {k: load_predictions(k, "val") for k in model_keys}
print("\nLoading test predictions...")
test_preds = {k: load_predictions(k, "test") for k in model_keys}

# ── Load train/val/test texts from PROCESSED parquet (cleaned, 5 classes)
train_df = pd.read_parquet(f"{PROCESSED_DIR}/train.parquet")
val_df   = pd.read_parquet(f"{PROCESSED_DIR}/val.parquet")
test_df  = pd.read_parquet(f"{PROCESSED_DIR}/test.parquet")

train_texts  = train_df["tweet_text"].tolist()
val_texts    = val_df["tweet_text"].tolist()[:len(val_preds["roberta"]["preds"])]
test_texts   = test_df["tweet_text"].tolist()[:len(test_preds["roberta"]["preds"])]

train_labels = train_df["label"].values.astype(int)
val_labels   = val_preds["roberta"]["labels"]
test_labels  = test_preds["roberta"]["labels"]

class_names = [
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "not_humanitarian",
    "other_relevant_information",
    "rescue_volunteering_or_donation_effort",
]

print(f"\ntrain: {len(train_texts)} samples, labels: {sorted(set(train_labels.tolist()))}")
print(f"val:   {len(val_texts)} samples")
print(f"test:  {len(test_texts)} samples")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: Model Characterisation (Tweet Styles)
# ══════════════════════════════════════════════════════════════════════════════
from model_characterisation import (
    classify_tweets_batch, compute_style_performance_matrix,
    plot_style_performance_heatmap,
)

# Classify tweet styles
val_style_labels, _ = classify_tweets_batch(val_texts)
test_style_labels, _ = classify_tweets_batch(test_texts)

print(f"Val style distribution:  {dict(pd.Series(val_style_labels).value_counts())}")
print(f"Test style distribution: {dict(pd.Series(test_style_labels).value_counts())}")

# Compute style performance matrix (for transformers)
display_names = {
    "roberta": "RoBERTa", "deberta": "DeBERTa", "electra": "ELECTRA",
    "bert": "BERT", "bertweet": "BERTweet", "xtremedistil": "XtremeDistil",
}
perf_matrix = compute_style_performance_matrix(
    {display_names[k]: val_preds[k] for k in display_names},
    val_style_labels,
    list(display_names.values()),
    save_dir=os.path.join(RESULTS_DIR, "model_style"),
)

EVAL_DIR = os.path.join(RESULTS_DIR, "evaluation_results")
os.makedirs(EVAL_DIR, exist_ok=True)
plot_style_performance_heatmap(
    perf_matrix, list(display_names.values()),
    save_path=os.path.join(EVAL_DIR, "model_style_heatmap.png"),
)
print("✅ Model characterisation complete.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: Dynamic Ensemble (Novelty 1)
# ══════════════════════════════════════════════════════════════════════════════
from dynamic_ensemble import (
    train_meta_learner, predict_with_dynamic_ensemble,
    evaluate_dynamic_ensemble, save_ensemble_artifacts,
)

# Build probability lists (ordered)
val_probs_list  = [val_preds[k]["probs"]  for k in model_keys]
test_probs_list = [test_preds[k]["probs"] for k in model_keys]

# Train meta-learner with style features
meta_learner, scaler, feature_names = train_meta_learner(
    val_model_probs_list=val_probs_list,
    val_labels=val_labels,
    val_tweet_texts=val_texts,
    val_style_labels=val_style_labels,
    meta_learner_type="mlp",
)

# Predict on test set
ensemble_probs, ensemble_preds = predict_with_dynamic_ensemble(
    meta_learner, scaler, test_probs_list, test_texts,
    test_style_labels=test_style_labels,
)

ensemble_metrics = evaluate_dynamic_ensemble(
    ensemble_preds, ensemble_probs, test_labels, class_names,
)

ENSEMBLE_DIR = os.path.join(RESULTS_DIR, "ensemble")
save_ensemble_artifacts(
    meta_learner, scaler, ensemble_probs, ensemble_preds,
    test_labels, ENSEMBLE_DIR,
)
print("✅ Dynamic ensemble complete.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Novelty 2 — Class-Adaptive Confidence Thresholds
# ══════════════════════════════════════════════════════════════════════════════
from adaptive_confidence import sweep_per_class_thresholds, apply_per_class_thresholds

# Generate ensemble predictions on validation set for threshold sweeping
val_ensemble_probs, val_ensemble_preds = predict_with_dynamic_ensemble(
    meta_learner, scaler, val_probs_list, val_texts,
    test_style_labels=val_style_labels,
)

# Sweep thresholds on validation set
per_class_thresholds = sweep_per_class_thresholds(
    ensemble_probs=val_ensemble_probs,
    ensemble_preds=val_ensemble_preds,
    true_labels=val_labels,
    class_names=class_names,
)

print("\nPer-class thresholds:")
for cls, t in per_class_thresholds.items():
    print(f"  {cls}: {t:.2f}")

# Apply thresholds to test set
selective_results = apply_per_class_thresholds(
    ensemble_probs=ensemble_probs,
    ensemble_preds=ensemble_preds,
    true_labels=test_labels,
    per_class_thresholds=per_class_thresholds,
    class_names=class_names,
)

confidence_accepted_mask = selective_results["accepted_mask"]

# Report
from sklearn.metrics import f1_score as sk_f1
full_f1 = sk_f1(test_labels, ensemble_preds, average="macro")
print(f"\nWithout selective prediction:  Macro F1 = {full_f1:.4f}")
print(f"With class-adaptive thresholds:")
print(f"  Coverage:  {selective_results['coverage']:.4f} ({selective_results['accepted_count']}/{selective_results['total']})")
print(f"  Macro F1:  {selective_results['macro_f1']:.4f}")
print(f"  Accuracy:  {selective_results['accuracy']:.4f}")
print(f"  F1 improvement: +{selective_results['macro_f1'] - full_f1:.4f}")

# Save confidence results
import json as json_mod
CONFIDENCE_DIR = os.path.join(RESULTS_DIR, "confidence_results")
os.makedirs(CONFIDENCE_DIR, exist_ok=True)
with open(os.path.join(CONFIDENCE_DIR, "selective_results.json"), "w") as f:
    json_mod.dump({k: v for k, v in selective_results.items() if k != "accepted_mask"}, f, indent=2)
with open(os.path.join(CONFIDENCE_DIR, "per_class_thresholds.json"), "w") as f:
    json_mod.dump(per_class_thresholds, f, indent=2)
print(f"\n✅ Confidence results saved to {CONFIDENCE_DIR}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: Novelty 3 — Attribution Filter (installs captum)
# ══════════════════════════════════════════════════════════════════════════════
import subprocess
subprocess.run(["pip", "install", "-q", "captum"], check=True)

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from attribution_filter import (
    compute_attributions_for_batch, flag_unreliable_predictions, combined_abstention
)

# Load RoBERTa from saved output (try seed_42 path first)
roberta_path = f"{OUTPUT_DIR}/roberta/seed_42/best_model"
if not os.path.exists(roberta_path):
    roberta_path = f"{OUTPUT_DIR}/roberta/best_model"
print(f"Loading RoBERTa from: {roberta_path}")

device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    tokenizer = AutoTokenizer.from_pretrained(roberta_path)
    model = AutoModelForSequenceClassification.from_pretrained(roberta_path)
    model.to(device).eval()
    print(f"✅ Loaded RoBERTa from {roberta_path}")
except Exception as e:
    print(f"❌ Failed to load RoBERTa: {e}")
    raise

N_SAMPLES = min(500, len(test_texts))
print(f"Computing attributions for {N_SAMPLES} samples...")
attribution_results = compute_attributions_for_batch(
    model=model, tokenizer=tokenizer,
    texts=test_texts[:N_SAMPLES],
    predicted_classes=ensemble_preds[:N_SAMPLES],
    device=device, n_steps=50,
)

reliability_mask_subset, reliability_details = flag_unreliable_predictions(
    attribution_results=attribution_results,
    probs=ensemble_probs[:N_SAMPLES], preds=ensemble_preds[:N_SAMPLES],
    class_names=class_names, per_class_thresholds=per_class_thresholds,
)

reliability_mask = np.ones(len(test_labels), dtype=bool)
reliability_mask[:N_SAMPLES] = reliability_mask_subset

combined_mask, combined_results = combined_abstention(
    confidence_accepted_mask=confidence_accepted_mask,
    reliability_mask=reliability_mask,
    preds=ensemble_preds, labels=test_labels, class_names=class_names,
)
print("✅ Attribution filter complete.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Comprehensive Evaluation Report
# ══════════════════════════════════════════════════════════════════════════════
from evaluation import run_full_evaluation

display_names_all = {
    "roberta": "RoBERTa", "deberta": "DeBERTa", "electra": "ELECTRA",
    "bert": "BERT", "bertweet": "BERTweet", "xtremedistil": "XtremeDistil",
    "cnn": "CNN", "bilstm": "BiLSTM",
}
model_test_results = {display_names_all[k]: test_preds[k] for k in model_keys}
ensemble_test_results = {"preds": ensemble_preds, "labels": test_labels, "probs": ensemble_probs}

run_full_evaluation(
    model_test_results=model_test_results,
    ensemble_test_results=ensemble_test_results,
    selective_results=selective_results,
    combined_results=combined_results,
    train_texts=train_texts, train_labels=train_labels,
    test_texts=test_texts, test_labels=test_labels,
    class_names=class_names,
    output_dir=os.path.join(RESULTS_DIR, "evaluation_results"),
    model_output_dir=OUTPUT_DIR,
    style_performance_path=os.path.join(RESULTS_DIR, "model_style", "model_style_performance.json"),
)

# Save summary JSON
summary = {
    "ensemble_macro_f1": float(sk_f1(test_labels, ensemble_preds, average="macro")),
    "selective_macro_f1": float(selective_results["macro_f1"]),
    "selective_coverage": float(selective_results["coverage"]),
    "combined_macro_f1": float(combined_results["combined_macro_f1"]),
    "combined_coverage": float(combined_results["combined_coverage"]),
}
with open(os.path.join(RESULTS_DIR, "final_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(f"\n✅ All results saved to {RESULTS_DIR}")
print(f"Summary: {json.dumps(summary, indent=2)}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Launch Gradio Dashboard
# ══════════════════════════════════════════════════════════════════════════════
from crisis_dashboard import create_dashboard

app = create_dashboard(
    model_dir=OUTPUT_DIR,
    label_mapping_path=f"{PROCESSED_DIR}/label_mapping.json",
    ensemble_dir=os.path.join(RESULTS_DIR, "ensemble"),
)

print("\n🚀 Launching Crisis Dashboard...")
app.launch(share=True, debug=False)
